In [ ]:
import warnings

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import VimeoVideo
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.utils.validation import check_is_fitted

warnings.simplefilter(action="ignore", category=FutureWarning)

In this project, you're working for a client who wants to create a model that can predict the price of apartments in the city of Buenos Aires — with a focus on apartments that cost less than $400,000 USD.

**Task 2.1.1: Write a function named wrangle that takes a file path as an argument and returns a DataFrame.**

In [1]:
import pandas as pd

def wrangle(filepath):
    # Import CSV
    df = pd.read_csv(filepath)

    # Subset to properties in "Capital Federal"
    mask_ba = df["place_with_parent_names"].str.contains("Capital Federal")
    mask_type = df["property_type"] == "apartment"

    # Subset to properties where "price_aprox_usd" < 400,000
    mask_price = df["price_aprox_usd"] < 400000

    df = df[mask_ba & mask_type & mask_price]

    return df

**Task 2.1.2: Use your wrangle function to create a DataFrame df from the CSV file data/buenos-aires-real-estate-1.csv.**

In [ ]:
df = wrangle("data/buenos-aires-real-estate-1.csv")
print("df shape:", df.shape)
df.head()

# Check your work


In [ ]:
assert (
    len(df) <= 8606
), f"`df` should have no more than 8606 observations, not {len(df)}."

or this project, we want to build a model for apartments in Buenos Aires proper ("Capital Federal") that cost less than $400,000. Looking at the first five rows of our DataFrame, we can already see that there properties that fall outside those parameters. So our first cleaning task is to remove those observations from our dataset. Since we're using a function to import and clean our data, we'll need to make changes there.

**Task 2.1.3: Add to your wrangle function so that the DataFrame it returns only includes apartments in Buenos Aires ("Capital Federal") that cost less than $400,000 USD. Then recreate df from data/buenos-aires-real-estate-1.csv by re-running the cells above.**

Subset a DataFrame with a mask using pandas.
To check your work, df should no have no more than 1,781 observations.

In [ ]:
import pandas as pd

def wrangle(filepath):
    # Import CSV
    df = pd.read_csv(filepath)

    # Subset to properties in "Capital Federal"
    mask_ba = df["place_with_parent_names"].str.contains("Capital Federal")
    mask_type = df["property_type"] == "apartment"

    # Subset to properties where "price_aprox_usd" < 400,000
    mask_price = df["price_aprox_usd"] < 400000

    df = df[mask_ba & mask_type & mask_price]

    return df

In [ ]:
# Check your work
assert (
    len(df) <= 1781
), f"`df` should have no more than 1781 observations, not {len(df)}."

We saw in the previous project that property size is an important factor in determining price. With that in mind, let's look at the distribution of apartment sizes in our dataset.

**Task 2.1.4: Create a histogram of "surface_covered_in_m2". Make sure that the x-axis has the label "Area [sq meters]" and the plot has the title "Distribution of Apartment Sizes".**

What's a histogram?
Create a histogram using Matplotlib.

In [ ]:
plt.hist(df["surface_covered_in_m2"])
plt.xlabel("Area [sq meters]")
plt.ylabel("Area [sq meters]")
plt.title("Distribution of Apartment Sizes");

Yikes! When you see a histogram like the one above, it suggests that there are outliers in your dataset. This can affect model performance — especially in the sorts of linear models we'll learn about in this project. To confirm, let's look at the summary statistics for the "surface_covered_in_m2" feature.

**Task 2.1.5: Calculate the summary statistics for df using the describe method.**

What's skewed data?
Print the summary statistics for a DataFrame using pandas.

In [ ]:
df.describe()["surface_covered_in_m2"]

The statistics above confirm what we suspected. While most of the apartments in our dataset are smaller that 73 square meters, there are some that are several thousand square meters. The best thing to do is to change our wrangle function and remove them from the dataset.

**Task 2.1.6: Add to your wrangle function so that it removes observations that are outliers in the "surface_covered_in_m2" column. Specifically, all observations should fall between the 0.1 and 0.9 quantiles for "surface_covered_in_m2".**

What's a quantile?
Calculate the quantiles for a Series in pandas.
Subset a DataFrame with a mask using pandas.
When you're done, don't forget to rerun all the cells above. Note how your histogram changes now that there are no outliers. At this point, df should have no more than 1,343 observations.

In [ ]:
def wrangle(filepath):
    # Import CSV
    df = pd.read_csv(filepath)

    # Subset to properties in "Capital Federal"
    mask_ba = df["place_with_parent_names"].str.contains("Capital Federal")

    # Subset to properties where "property_type" == "apartment"
    mask_apt = df["property_type"] == "apartment"

    # Subset to properties where "price_aprox_usd" < 400,000
    mask_price = df["price_aprox_usd"] < 400,000

    df = df[mask_ba & mask_apt & mask_price]

    # Remove outliers: "surface_covered_in_m2"
    low, high = df["surface_covered_in_m2"].quantile([0.1, 0.9])
    mask_area = df["surface_covered_in_m2"].between(low, high)

    df = df[mask_area]

    return df

In [ ]:
# Check your work
assert len(df) <= 1343

**Task 2.1.7: Create a scatter plot that shows price ("price_aprox_usd") vs area ("surface_covered_in_m2") in our dataset. Make sure to label your x-axis "Area [sq meters]" and your y-axis "Price [USD]".**

What's a scatter plot?
Create a scatter plot using Matplotlib.

In [ ]:
plt.scatter(df["surface_covered_in_m2"], df["price_aprox_usd"])
plt.xlabel("Area [sq meters]")
plt.ylabel("Price [USD]")
plt.title("Buenos Aires: Price vs. Area");

Split
A key part in any model-building project is separating your target (the thing you want to predict) from your features (the information your model will use to make its predictions). Since this is our first model, we'll use just one feature: apartment size.

**Task 2.1.8: Create the feature matrix named X_train, which you'll use to train your model. It should contain one feature only: ["surface_covered_in_m2"]. Remember that your feature matrix should always be two-dimensional.**

What's a feature matrix?
Create a DataFrame from a Series in pandas.

In [ ]:
features = ["surface_covered_in_m2"]
X_train = df[features]
X_train.head()

# Check your work
assert X_train.shape == (
    1343,
    1,
), f"The shape of `X_train` should be (1343, 1), not {X_train.shape}."

**Task 2.1.9: Create the target vector named y_train, which you'll use to train your model. Your target should be "price_aprox_usd". Remember that, in most cases, your target vector should be one-dimensional.**

What's a target vector?
Select a Series from a DataFrame in pandas.

In [ ]:
target = "price_aprox_usd"
y_train = df[target]
y_train.shape

In [ ]:
# Check your work
assert y_train.shape == (1343,)

***Build Model***

Baseline
The first step in building a model is baselining. To do this, ask yourself how you will know if the model you build is performing well?" One way to think about this is to see how a "dumb" model would perform on the same data. Some people also call this a naïve or baseline model, but it's always a model makes only one prediction — in this case, it predicts the same price regardless of an apartment's size. So let's start by figuring out what our baseline model's prediction should be.

In [ ]:
y_mean = y_train.mean()
y_mean

**Task 2.1.10: Calculate the mean of your target vector y_train and assign it to the variable y_mean.**

What's a regression problem?
Calculate summary statistics for a DataFrame or Series in pandas.

In [ ]:
y_pred_baseline = [y_mean]* len(y_train)
len(y_pred_baseline)


**Task 2.1.11: Create a list named y_pred_baseline that contains the value of y_mean repeated so that it's the same length at y.**

Calculate the length of a list in Python.

In [ ]:
plt.plot(X_train.values, y_pred_baseline, color="orange", label="Baseline Model")
plt.scatter(X_train, y_train)
plt.xlabel("Area [sq meters]")
plt.ylabel("Price [USD]")
plt.title("Buenos Aires: Price vs. Area")
plt.legend();

Looking at this visualization, it seems like our baseline model doesn't really follow the trend in the data. But, as a data scientist, you can't depend only on a subjective plot to evaluate a model. You need an exact, mathematically calculate performance metric. There are lots of performance metrics, but the one we'll use here is the mean absolute error.

**Task 2.1.12: Add a line to the plot below that shows the relationship between the observations X_train and our dumb model's predictions y_pred_baseline. Be sure that the line color is orange, and that it has the label "Baseline Model".**

What's a line plot?
Create a line plot in Matplotlib.

In [ ]:
plt.plot(X_train.values, y_pred_baseline, color="orange", label="Baseline Model")
plt.scatter(X_train, y_train)
plt.xlabel("Area [sq meters]")
plt.ylabel("Price [USD]")
plt.title("Buenos Aires: Price vs. Area")
plt.legend();

**Task 2.1.13: Calculate the baseline mean absolute error for your predictions in y_pred_baseline as compared to the true targets in y.**

What's a performance metric?
What's mean absolute error?
Calculate the mean absolute error for a list of predictions in scikit-learn.

In [ ]:
mae_baseline = mean_absolute_error(y_train ,y_pred_baseline)

print("Mean apt price", round(y_mean, 2))
print("Baseline MAE:", round(mae_baseline, 2))

**Iterate**

The next step in building a model is iterating. This involves building a model, training it, evaluating it, and then repeating the process until you're happy with the model's performance. Even though the model we're building is linear, the iteration process rarely follows a straight line. Be prepared for trying new things, hitting dead-ends, and waiting around while your computer does long computations to train your model. ☕️ Let's get started!

The first thing we need to do is create our model — in this case, one that uses linear regression.

**Task 2.1.14: Instantiate a LinearRegression model named model.**

What's linear regression?
What's a cost function?
Instantiate a predictor in scikit-learn.

In [ ]:
model = LinearRegression()

In [ ]:
# Check your work
assert isinstance(model, LinearRegression)

**Task 2.1.15: Fit your model to the data, X_train and y_train.**

Fit a model to training data in scikit-learn.

In [ ]:
model.fit(X_train,y_train)

In [ ]:
Check your work
check_is_fitted(model)

In [ ]:
# Check your work
assert (
    len(y_pred_training) == 1343
), f"There should be 1343 predictions in `y_pred_training`, not {len(y_pred_training)}."

**Task 2.1.16: Using your model's predict method, create a list of predictions for the observations in your feature matrix X_train. Name this array y_pred_training.**

Generate predictions using a trained model in scikit-learn.

In [ ]:
y_pred_training = model.predict(X_train)
y_pred_training[:5]

**Task 2.1.17: Calculate your training mean absolute error for your predictions in y_pred_training as compared to the true targets in y_train**.

Calculate the mean absolute error for a list of predictions in scikit-learn.

In [ ]:
mae_training = mean_absolute_error(y_train,y_pred_training)
print("Training MAE:", round(mae_training, 2))

Good news: Our model beat the baseline by over $10,000! That's a good indicator that it will be helpful in predicting apartment prices. But the real test is how the model performs on data that it hasn't seen before, data that we call the test set. In the future, you'll create your own test set before you train your model, but here we'll use one that's pre-made, and we'll evaluate the model using the WQU auto-grader.

**Task 2.1.18: Run the code below to import your test data buenos-aires-test-features.csv into a DataFrame and generate a Series of predictions using your model. After the code runs successfully, click the Check Activity button on the left pane to verify your predictions.**
**bold text**
What's generalizability?
Generate predictions using a trained model in scikit-learn.
Calculate the mean absolute error for a list of predictions in scikit-learn.

In [ ]:
X_test = pd.read_csv("data/buenos-aires-test-features.csv")[features]
y_pred_test = pd.Series(model.predict(X_test))
y_pred_test.head()

Once your model is built and tested, it's time to share it with others. If you're presenting to simple linear model to a technical audience, they might appreciate an equation. When we created our baseline model, we represented it as a line. The equation for a line like this is usually written as:

Equation: y = m*x + b

Since data scientists often work with more complicated linear models, they prefer to write the equation as:

Equation: y = beta 0 + beta 1 * x

Regardless of how we write the equation, we need to find the values that our model has determined for the intercept and and coefficient. Fortunately, all trained models in scikit-learn store this information in the model itself. Let's start with the intercept.

Task 2.1.19: Extract the intercept from your model, and assign it to the variable intercept.
**bold text**
What's an intercept in a linear model?
Access an attribute of a trained model in scikit-learn.

In [ ]:
intercept =round( model.intercept_)
print("Model Intercept:", intercept)
assert any([isinstance(intercept, int), isinstance(intercept, float)])

**Task 2.1.20: Extract the coefficient associated "surface_covered_in_m2" in your model, and assign it to the variable coefficient.**

What's a coefficient in a linear model?
Access an attribute of a trained model in scikit-learn.

In [ ]:
coefficient = round(model.coef_[0],2)
print('Model coefficient for "surface_covered_in_m2":', coefficient)
assert any([isinstance(coefficient, int), isinstance(coefficient, float)])

**Task 2.1.21: Complete the code below and run the cell to print the equation that your model has determined for predicting apartment price based on size.**

What's an f-string?

In [ ]:
print(f"apartment_price = {intercept} + {coefficient} * surface_covered")

**Task 2.1.22: Add a line to the plot below that shows the relationship between the observations in X_train and your model's predictions y_pred_training. Be sure that the line color is red, and that it has the label "Linear Model".**

In [ ]:
plt.plot(X_train.values, model.predict(X_train), color="magenta", label="Linear Model")
plt.scatter(X_train, y_train)
plt.xlabel("surface_covered [sq meters]")
plt.ylabel("price [USD]")
plt.legend();